# PyCMF tutorial

Cluster mean-field (cMF) theory in the AO basis, on top of PySCF. The wavefunction is a product of cluster states,
$|\Psi\rangle = |I\rangle\otimes|J\rangle\otimes\cdots$, built from one set of orbitals; everything between the clusters
is a mean field evaluated with the J/K builder of the PySCF mean-field object, so exact, in-core and density-fitted
integrals all work and the integrals of the full active space are never formed.

This notebook shows the main functions on small molecules, so it runs in a few minutes:

1. A first calculation, and the orbital-optimized cMF
2. Clusters from atomic orbitals (open-shell clusters, RO-cMF)
3. Orbital spaces by hand
4. State averaging (mixed valence)
5. RDMs and properties
6. Perturbation theory between clusters (QDPT2), its RDMs, and the many-body expansion
7. Dynamic correlation of the environment (cMF-NEVPT2)
8. Sub-problems for TPSCI in FermiCG.jl
9. Nuclear gradients and geometry optimization
10. Pair-density functional theory (cMF-PDFT) and exchange couplings
11. Larger systems

In [1]:
import numpy as np
from pyscf import gto, scf, mcscf, lib
import pycmf
from pycmf import pdft

np.set_printoptions(precision=6, suppress=True, linewidth=120)
cm = 219474.6313705     # Hartree -> cm-1

## 1. A first calculation: H$_6$ in Löwdin orbitals

`set_Cact(C, clusters, fspace)` gives the active orbitals, which of them form each cluster, and the number of
$\alpha$ and $\beta$ electrons of each cluster. `cmf_ci()` solves the cluster states self-consistently at fixed orbitals
(each cluster exactly, in the mean field of the others); `kernel()` also optimizes the orbitals. These are the numbers
of the H$_6$ test of ClusterMeanField.jl.

In [2]:
mol = gto.M(atom=[['H', (i, 0, 0)] for i in range(6)], basis='sto-3g', verbose=0)
mf = scf.RHF(mol).run()

cmf = pycmf.CMF(mf)
cmf.set_Cact(cmf.lowdin(), [[0, 1], [2, 3], [4, 5]], [(1, 1)] * 3)

e_ci = cmf.cmf_ci()
e_oo = cmf.kernel()
print('RHF          %.10f' % mf.e_tot)
print('CMF-CI       %.10f   (ClusterMeanField.jl: -2.9729381365)' % e_ci)
print('ooCMF        %.10f   (ClusterMeanField.jl: -3.2059830330)  converged: %s' % (e_oo, cmf.converged))

 Form lowdin orthogonalized orbitals


RHF          -3.1355322140
CMF-CI       -2.9729381365   (ClusterMeanField.jl: -2.9729381365)
ooCMF        -3.2059830330   (ClusterMeanField.jl: -3.2059830330)  converged: True


Settings of the orbital optimization (L-BFGS with an exact gradient) and of CMF-CI are attributes:
`conv_tol` (energy), `conv_tol_grad` (orbital gradient), `max_cycle`, `max_stepsize`, `conv_tol_rdm1`, `ci_conv_tol`, ...
Set `cmf.verbose = 4` to follow the iterations.

In [3]:
print('energy change for convergence:', cmf.conv_tol, ' orbital gradient:', cmf.conv_tol_grad)
print('cluster energies in their embedding:', np.array(cmf.cluster_energies))

energy change for convergence: 1e-10  orbital gradient: 1e-06
cluster energies in their embedding: [-3.094027 -3.065854 -3.094027]


## 2. Clusters from atomic orbitals: two high-spin N atoms

`set_clusters_from_ao(labels, fspace)` takes the active orbitals of each cluster from meta-Löwdin orthogonalized AOs,
and the doubly occupied and virtual orbitals from the mean-field density in the rest; no orbitals have to be localized.
Each cluster can have its own charge and spin: here two N atoms, each with three unpaired electrons, with antiparallel
$M_s$. Open-shell clusters are treated as in restricted open-shell cMF (RO-cMF): each cluster is an ensemble over the
$M_s$ components of its multiplet, so the energy and the orbitals are those of the whole spin ladder of the two atoms
(which QDPT2 resolves, section 6).

In [4]:
mol = gto.M(atom='N 0 0 0; N 0 0 2.5', basis='6-31g', spin=6, verbose=0)
mf = scf.ROHF(mol).run()

cmf_nn = pycmf.CMF(mf)
cmf_nn.set_clusters_from_ao([['0 N 2p'], ['1 N 2p']], [(3, 0), (0, 3)])
cmf_nn.kernel()
print('ROHF (S = 3)  %.10f' % mf.e_tot)
print('RO-cMF        %.10f' % cmf_nn.e_tot)
ss, multip, ss_clusters = cmf_nn.spin_square()
print('<S_I^2> of the clusters:', np.round(ss_clusters, 6), ' product state <S^2> =', round(ss, 6))

ROHF (S = 3)  -108.7534842181
RO-cMF        -108.7514777082


<S_I^2> of the clusters: [3.75 3.75]  product state <S^2> = 3.0


In [5]:
pop_and_charges, dipole = cmf_nn.analyze(verbose=4)     # occupations, <S_I^2>, charges and spin populations

CMF energy = -108.751477708220


cluster  (na, nb)      N       <S_I^2>   E (cluster in its embedding)


    0    (3, 0)     3.0000     3.7500   -108.1665232489


         natural occupations: 1.0000 1.0000 1.0000


    1    (0, 3)     3.0000     3.7500   -108.1665232500


...

In total             7.00000 | 7.00000   


 ** Mulliken atomic charges   ( Nelec_alpha | Nelec_beta ) **


charge of  0N =      0.00000  (     4.99797      2.00203 )


charge of  1N =     -0.00000  (     2.00203      4.99797 )


Dipole moment(X, Y, Z, Debye): -0.00000, -0.00000, -0.00000


## 3. Orbital spaces by hand

`set_Cdoc`, `set_Cact` and `set_Cvir` take any orbitals (frozen orbitals, not optimized: `set_Cfrz`). With a single
cluster, CMF-CI is CASCI and the orbital-optimized cMF is CASSCF.

In [6]:
mol = gto.M(atom='N 0 0 0; N 0 0 1.3', basis='6-31g', verbose=0)
mf = scf.RHF(mol).run()
C = mf.mo_coeff

cas = pycmf.CMF(mf)
cas.set_Cdoc(C[:, :4])
cas.set_Cact(C[:, 4:10], [list(range(6))], [(3, 3)])
cas.set_Cvir(C[:, 10:])
print('CMF-CI, one cluster %.10f' % cas.cmf_ci())
print('CASCI(6,6)          %.10f' % mcscf.CASCI(mf, 6, 6).kernel()[0])

CMF-CI, one cluster -108.9072285956


CASCI(6,6)          -108.9072285956


## 4. State averaging: a mixed-valence dimer

In (H$_2$)$_2^+$ the hole can sit on either H$_2$. `state_average_` optimizes one set of orbitals for the average of
both localized configurations; `select_state(k)` makes state `k` current for the RDMs, and QDPT2 then mixes the two
configurations (the delocalized states and their splitting).

In [7]:
mol = gto.M(atom='H 0 0 0; H 0 0 .74; H 3 0 0; H 3 0 .74', basis='6-31g', charge=1, spin=1, verbose=0)
mf = scf.ROHF(mol).run()

mv = pycmf.CMF(mf)
mv.set_clusters_from_ao([['0 H', '1 H'], ['2 H', '3 H']], [(1, 1), (1, 0)])
mv.state_average_([[(1, 1), (1, 0)], [(1, 0), (1, 1)]])
mv.kernel()
S = mf.get_ovlp()
for k in range(2):
    mv.select_state(k)
    dm_a, dm_b = mv.make_rdm1s(ao_repr=True)
    spin = np.einsum('ij,ji->i', dm_a - dm_b, S)            # Mulliken spin population per AO
    labels = mol.ao_labels(fmt=False)
    on = [sum(spin[i] for i, l in enumerate(labels) if l[0] in atoms) for atoms in ((0, 1), (2, 3))]
    print('state %d  E = %.8f  spin on H2(A), H2(B): %.3f %.3f' % (k, mv.e_states[k], *on))
mv.select_state(0)
res = mv.qdpt2()
print('QDPT2 states: %s  splitting %.2f mEh' % (np.round(res.energies, 8), 1000*(res.energies[1] - res.energies[0])))

state 0  E = -1.70841747  spin on H2(A), H2(B): 0.000 1.000
state 1  E = -1.70841747  spin on H2(A), H2(B): 1.000 0.000


QDPT2 states: [-1.716774 -1.700739]  splitting 16.03 mEh


## 5. RDMs and properties

| method | returns |
| --- | --- |
| `make_rdm1(ao_repr=True)` | spin-summed 1RDM of the whole system in the AO basis |
| `make_rdm1s(ao_repr=True)` | $\alpha$, $\beta$ 1RDMs of the product state, each cluster in its `fspace` $M_s$ sector (`spin_average=True`: RO-cMF ensemble) |
| `make_rdm1()`, `make_rdm12()` | active-space 1RDM and spin-free 2RDM (PySCF convention), consistent with `e_tot` |
| `make_rdm12s()` | spin-resolved active-space RDMs of the product state |
| `spin_square()`, `analyze()` | $\langle S^2\rangle$, populations, dipole |

In [8]:
dm = cmf_nn.make_rdm1(ao_repr=True)
print('electrons:', np.trace(dm @ cmf_nn.S).round(8))
print('dipole (Debye):', cmf_nn.mf.dip_moment(cmf_nn.mol, dm, verbose=0).round(6))

# the energy from the active-space RDMs and the active-space Hamiltonian is e_tot
h0, h1, h2, orbs = cmf_nn.get_active_hamiltonian()
dm1, dm2 = cmf_nn.make_rdm12()
idx = np.ix_(orbs, orbs)
e = h0 + np.einsum('pq,pq', h1, dm1[idx]) + .5*np.einsum('pqrs,pqrs', h2, dm2[np.ix_(orbs, orbs, orbs, orbs)])
print('E from RDMs - e_tot = %.1e' % (e - cmf_nn.e_tot))

# orbitals for visualization: pyscf.tools.molden.from_mo(mol, 'cmf.molden', cmf_nn.mo_coeff)

electrons: 14.0
dipole (Debye): [-0.       -0.       -0.000001]
E from RDMs - e_tot = 0.0e+00


## 6. Perturbation theory between clusters (QDPT2)

`qdpt2(clusters)` treats a few clusters (default: all) in the mean field of the rest. The model space contains all
$M_s$ components of the cluster multiplets (and, for state averaging, the other configurations), so spin coupling is
exact and charge transfer is second order. For the two N atoms this gives the spin ladder $S = 0,\dots,3$ and the
exchange coupling $J$ of $H = -2J\,\mathbf{S}_A\cdot\mathbf{S}_B$ (FCI over the same orbitals: $-190$ cm$^{-1}$).

In [9]:
res = cmf_nn.qdpt2()
for e, s in zip(res.energies, res.spins):
    print('S = %.0f   E = %.10f' % (s, e))
print('J = %.1f cm-1' % (res.heisenberg_j()*cm))

S = 0   E = -108.7633977444
S = 1   E = -108.7617161898
S = 2   E = -108.7583399234
S = 3   E = -108.7532420355
J = -185.8 cm-1


### RDMs of the QDPT2 wavefunction

With `with_rdm=True` the first-order wavefunction $\Psi_0 + \Psi_1$ of every QDPT2 state is kept, so its properties
include the coupling between the clusters: charge transfer at first order and the exact spin coupling. The RDMs are
those of a wavefunction (N-representable), and spin eigenstates stay spin pure. `order=0` gives the model-space part.

In [10]:
res = cmf_nn.qdpt2(with_rdm=True)
for k in range(len(res.energies)):
    dm_a, dm_b = res.make_rdm1s(state=k, ao_repr=True)
    print('state %d: <S^2> = %.6f   spin density integrates to %.1e' % (k, res.spin_square(k), np.trace((dm_a - dm_b) @ cmf_nn.S)))
dm1, dm2 = res.make_rdm12(state=0)          # in the orbitals res.orbs of the selected clusters
pop_and_charges, dipole = res.analyze(state=0, verbose=4)

state 0: <S^2> = 0.000000   spin density integrates to -3.7e-16
state 1: <S^2> = 2.000000   spin density integrates to -1.9e-16
state 2: <S^2> = 6.000000   spin density integrates to 5.4e-16
state 3: <S^2> = 12.000000   spin density integrates to 2.2e-16


QDPT2 state 0: E = -108.763397531545  <S^2> = 0.000000  (first-order wavefunction)


 ** Mulliken pop alpha/beta on meta-lowdin orthogonal AOs **


 ** Mulliken pop       alpha | beta **


pop of  0 N 1s        1.00000 | 1.00000   


...

In total             7.00000 | 7.00000   


 ** Mulliken atomic charges   ( Nelec_alpha | Nelec_beta ) **


charge of  0N =     -0.00000  (     3.50000      3.50000 )


charge of  1N =      0.00000  (     3.50000      3.50000 )


Dipole moment(X, Y, Z, Debye): -0.00000, -0.00000,  0.00001


### Many-body expansion over clusters

For many clusters, `pt2_mbe(order, cutoff)` adds the QDPT2 corrections of pairs and triples of clusters (each in the
mean field of the rest); `pair_pt2` is the pair truncation. With three clusters, `order=3` is exact.
`with_rdm=True` expands the 1RDM in the same way. Hexatriene, its $\pi$ system in three clusters:

In [11]:
atoms = [['C', (0.000, 0.000, 0)], ['C', (1.169, 0.675, 0)], ['C', (2.425, -0.050, 0)],
         ['C', (3.594, 0.625, 0)], ['C', (4.850, -0.100, 0)], ['C', (6.019, 0.575, 0)],
         ['H', (-0.944, 0.545, 0)], ['H', (0.000, -1.090, 0)], ['H', (1.169, 1.765, 0)],
         ['H', (2.425, -1.140, 0)], ['H', (3.594, 1.715, 0)], ['H', (4.850, -1.190, 0)],
         ['H', (6.963, 0.030, 0)], ['H', (6.019, 1.665, 0)]]
mol = gto.M(atom=atoms, basis='sto-3g', verbose=0)
mf = scf.RHF(mol).density_fit().run()
hex3 = pycmf.CMF(mf)
hex3.set_clusters_from_ao([['0 C 2pz', '1 C 2pz'], ['2 C 2pz', '3 C 2pz'], ['4 C 2pz', '5 C 2pz']], [(1, 1)] * 3)
hex3.kernel()

e_full = hex3.qdpt2().energies[0]
e2, inc2 = hex3.pt2_mbe(order=2)
e3, inc3, dm1, dm1_ao = hex3.pt2_mbe(order=3, with_rdm=True)
print('cMF                 %.8f' % hex3.e_tot)
print('QDPT2, all clusters %.8f' % e_full)
print('pairs               %.8f   (error %.2e)' % (e2, e2 - e_full))
print('pairs + triple      %.8f   (error %.2e)' % (e3, e3 - e_full))
print('increments:', {k: round(v, 6) for k, v in inc3.items()})
print('pairs within 3 A:', sorted(hex3.pt2_mbe(order=2, cutoff=3.0)[1]))

cMF                 -229.07129302
QDPT2, all clusters -229.08136192
pairs               -229.08108121   (error 2.81e-04)
pairs + triple      -229.08136192   (error 0.00e+00)
increments: {(0, 1): np.float64(-0.004765), (0, 2): np.float64(-0.000257), (1, 2): np.float64(-0.004765), (0, 1, 2): np.float64(-0.000281)}


pairs within 3 A: [(0, 1), (1, 2)]


## 7. Dynamic correlation of the environment: cMF-NEVPT2

`nevpt2()` adds strongly contracted NEVPT2 for the excitations involving the doubly occupied and virtual orbitals. Its
zeroth-order Hamiltonian has the embedded cluster Hamiltonians as active part, the same $H_0$ as QDPT2, so the two
corrections add: $E = E_{\rm cMF} + (E_{\rm QDPT2} - E_{\rm cMF}) + E_{\rm NEVPT2}$. For active spaces too large for
the reference RDMs, `nevpt2_mbe(order, cutoff)` expands it over clusters, pairs and triples. With density fitting the
integrals of all terms are sliced from shared three-index integrals.

In [12]:
e_nev, parts = hex3.nevpt2()
print('cMF-NEVPT2 correlation %.8f' % e_nev)
print({k: round(v, 6) for k, v in parts.items()})
for order in (1, 2, 3):
    e, inc = hex3.nevpt2_mbe(order=order)
    print('expansion order %d: %.8f  (error %.1e, %d terms)' % (order, e, e - e_nev, len(inc)))
print('E(cMF) + QDPT2 + NEVPT2 = %.8f' % (e_full + e_nev))

cMF-NEVPT2 correlation -0.26197114
{'Sr': np.float64(-0.0), 'Si': np.float64(-0.0), 'Sijrs': np.float64(-0.16675), 'Sijr': np.float64(0.0), 'Srsi': np.float64(0.0), 'Srs': np.float64(-0.001391), 'Sij': np.float64(-0.003), 'Sir': np.float64(-0.090831)}


expansion order 1: -0.25320255  (error 8.8e-03, 4 terms)


expansion order 2: -0.26196145  (error 9.7e-06, 7 terms)


expansion order 3: -0.26197114  (error 3.3e-16, 8 terms)
E(cMF) + QDPT2 + NEVPT2 = -229.34333305


## 8. Sub-problems for TPSCI in FermiCG.jl

`get_active_hamiltonian(clusters)` folds the core and the other clusters into the mean field, so solving it exactly
adds the correlation within the selected clusters; `export_fermicg(file, clusters)` writes it, with the clusters, the
reference occupations and the cMF RDMs, for FermiCG.jl (see the README for the Julia side).

In [13]:
import os, tempfile
h0, h1, h2, orbs = hex3.get_active_hamiltonian([0, 1])
print('pair (0, 1): %d orbitals, h0 = %.6f' % (len(orbs), h0))
path = os.path.join(tempfile.mkdtemp(), 'hexatriene_pair.npz')
hex3.export_fermicg(path, clusters=[0, 1])
print(sorted(np.load(path).files))

pair (0, 1): 4 orbitals, h0 = -225.806605


['cluster_index', 'e_cmf', 'e_states', 'fspace', 'fspace_states', 'h0', 'h1', 'h2', 'mo_coeff', 'nact', 'ncore', 'rdm1a', 'rdm1b', 'weights']


## 9. Nuclear gradients and geometry optimization

Orbital-optimized cMF is variational in its orbitals and cluster states, so `nuc_grad()` needs no response equations
(exact or density-fitted integrals; for state averaging, the gradient of the averaged energy). Here it is checked against
finite differences for H$_4$ in two clusters.

In [14]:
def h4(xyz):
    mf = scf.RHF(gto.M(atom=[['H', tuple(r)] for r in xyz], basis='6-31g', unit='Bohr', verbose=0)).run()
    c = pycmf.CMF(mf)
    c.set_clusters_from_ao([['0 H', '1 H'], ['2 H', '3 H']], [(1, 1), (1, 1)])
    c.conv_tol_grad = 1e-7
    c.kernel()
    return c

xyz = gto.M(atom='H 0 0 0; H 0.1 0 0.9; H 0 1.6 0.2; H 0.2 1.7 1.1', verbose=0).atom_coords()
g = h4(xyz).nuc_grad(verbose=0)
h = 1e-4
dx = np.zeros_like(xyz); dx[0, 2] = h
fd = (h4(xyz + dx).e_tot - h4(xyz - dx).e_tot) / (2*h)
print('dE/dz(H0): analytic %.8f  finite difference %.8f' % (g[0, 2], fd))

dE/dz(H0): analytic -0.07034268  finite difference -0.07034269


`nuc_grad_method().as_scanner()` gives energies and gradients at new geometries (the SCF is rerun, the clusters are
rebuilt from the same AO labels, and the orbitals start from the previous geometry), so the PySCF optimizers accept the
`CMF` object. Butadiene, its $\pi$ system in two clusters (requires `pyberny` or `geomeTRIC`):

In [15]:
mol = gto.M(atom="""C -1.8 0.65 0; C -0.67 -0.05 0; C 0.67 0.05 0; C 1.8 -0.65 0
H -2.75 0.12 0; H -1.80 1.73 0; H -0.67 -1.13 0; H 0.67 1.13 0; H 2.75 -0.12 0; H 1.80 -1.73 0""",
            basis='sto-3g', verbose=0)
bd = pycmf.CMF(scf.RHF(mol).density_fit().run())
bd.set_clusters_from_ao([['0 C 2pz', '1 C 2pz'], ['2 C 2pz', '3 C 2pz']], [(1, 1), (1, 1)])
try:
    from pyscf.geomopt import berny_solver
    mol_eq = berny_solver.optimize(bd)
    r = mol_eq.atom_coords(unit='Angstrom')
    print('C=C %.4f A   C-C %.4f A' % (np.linalg.norm(r[0] - r[1]), np.linalg.norm(r[1] - r[2])))
except ImportError:
    print('install pyberny (pip install pyberny) or geomeTRIC for geometry optimization')

C=C 1.3417 A   C-C 1.5070 A


## 10. Pair-density functional theory: cMF-PDFT and exchange couplings

`pdft.energy(cmf, otxc)` evaluates MC-PDFT, $E = V_{nn} + {\rm tr}(Dh) + \tfrac12{\rm tr}(DJ[D]) + E_{\rm ot}[\rho,\Pi]$,
with the density and on-top pair density of a cMF state and the on-top functionals of `pyscf.mcpdft` (tPBE, ftPBE,
tBLYP, ...). With one cluster it is MC-PDFT of CASCI.

For two spin-coupled clusters, `pdft.spin_ladder` uses the closed-form 2RDM of the states $|S_A S_B; S\rangle$ (no
determinant space). These states have no charge-transfer configurations, so no kinetic exchange, and their PDFT $J$ is
ferromagnetic. The QDPT2 first-order wavefunction has them: `pdft.qdpt2_spin_ladder` (MC-PDFT on the full CASCI
spin states over the same orbitals: $J = -270$ cm$^{-1}$).

In [16]:
ot = pdft.make_ot(cmf_nn.mol, 'tPBE', grids_level=3)       # build the grid once, reuse it
e_pdft, e_ot = pdft.energy(cmf_nn, ot=ot)
print('cMF-PDFT (tPBE) of the RO-cMF state %.8f   E_ot = %.8f' % (e_pdft, e_ot))

e0, j0 = pdft.spin_ladder(cmf_nn, ot=ot)
res = cmf_nn.qdpt2(with_rdm=True)
e1, j1 = pdft.qdpt2_spin_ladder(cmf_nn, res, order=1, ot=ot)
print('J (cm-1): QDPT2 %.0f | cMF-PDFT, spin-coupled states %.0f | cMF-PDFT, QDPT2 first order %.0f'
      % (res.heisenberg_j()*cm, j0*cm, j1*cm))

cMF-PDFT (tPBE) of the RO-cMF state -109.00760480   E_ot = -13.46311945


J (cm-1): QDPT2 -186 | cMF-PDFT, spin-coupled states 74 | cMF-PDFT, QDPT2 first order -257


## 11. Larger systems

* Use density fitting, `scf.RHF(mol).density_fit()`: the 3-index integrals, $8N_{\rm aux}N_{\rm AO}^2/2$ bytes, have to fit
  in memory (`mol.max_memory`). Every orbital step costs two J/K builds plus one pass for the cluster integrals,
  independent of the number of clusters; with density fitting the J/K builds use the orbitals, so K costs
  $O(N_{\rm aux}N_{\rm AO}^2N_{\rm occ})$.
* The README has timings: C$_{100}$H$_{102}$ (602 AOs, 25 clusters, 100 active orbitals) and C$_{40}$H$_{42}$ with QDPT2 and NEVPT2.
* For many clusters use `pt2_mbe` and `nevpt2_mbe` with a `cutoff`, and freeze the core for NEVPT2 (`set_Cfrz`).

A polyene with 16 carbons and four clusters of four $\pi$ orbitals:

In [17]:
import time
n = 16
atoms = []
for i in range(n):
    x, y = 1.24*i, (0.36 if i % 2 else -0.36)
    atoms.append(['C', (x, y, 0)])
    atoms.append(['H', (x, y + (1.08 if i % 2 else -1.08), 0)])
atoms += [['H', (-1.0, -0.9, 0)], ['H', (1.24*(n-1) + 1.0, 0.9, 0)]]
mol = gto.M(atom=atoms, basis='sto-3g', verbose=0)
t = time.time()
mf = scf.RHF(mol).density_fit().run()
poly = pycmf.CMF(mf)
poly.set_clusters_from_ao([['%d C 2pz' % (2*a) for a in range(k, k + 4)] for k in range(0, n, 4)], [(2, 2)] * (n//4))
poly.kernel()
print('%d AOs, %d clusters, %d active orbitals: ooCMF %.8f in %.0f s' % (mol.nao, len(poly.clusters), poly.Cact.shape[1], poly.e_tot, time.time() - t))
t = time.time()
e, inc = poly.pt2_mbe(order=2, cutoff=6.0)
print('pair QDPT2 within 6 A (%d pairs): %.8f in %.0f s' % (len(inc), e, time.time() - t))

98 AOs, 4 clusters, 16 active orbitals: ooCMF -608.62525834 in 209 s


pair QDPT2 within 6 A (3 pairs): -608.65391705 in 4 s
